## 1. 데이터 로딩 및 검증

Fragrantica 향수 콘텐츠 데이터와 Atrafshan 사용자 평점 데이터를 불러온다.

각 데이터셋의 크기, 주요 컬럼, 결측값을 확인하고
향수 ID 매핑에 필요한 데이터가 준비되었는지 점검한다.

In [1]:
from pathlib import Path
import pandas as pd

# 현재 실행 위치 확인
print("현재 실행 위치:", Path.cwd())

# 프로젝트의 data 폴더 찾기
if (Path.cwd() / "data").is_dir():
    DATA_DIR = Path.cwd() / "data"
elif (Path.cwd().parent / "data").is_dir():
    DATA_DIR = Path.cwd().parent / "data"
else:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다.")

# 데이터셋 로딩
perfumes = pd.read_csv(DATA_DIR / "perfumes_content.csv")
ratings = pd.read_csv(DATA_DIR / "user_ratings.csv")
linked = pd.read_csv(DATA_DIR / "user_ratings_linked.csv")
mapping = pd.read_csv(DATA_DIR / "perfume_mapping.csv")
atrafshan = pd.read_csv(DATA_DIR / "atrafshan_perfumes.csv")

# 데이터셋 확인
datasets = {
    "Fragrantica 콘텐츠": perfumes,
    "Atrafshan 평점": ratings,
    "연결된 평점": linked,
    "향수 ID 매핑": mapping,
    "Atrafshan 향수": atrafshan
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,}행, {len(df.columns)}열")

print("\n평점 데이터 컬럼:")
print(ratings.columns.tolist())

print("\n평점 데이터 예시:")
display(ratings.head())

print("\n결측값 확인:")
print(ratings[["user_id", "perfume_id", "scent_rating"]].isna().sum())

현재 실행 위치: c:\Users\sumin\perfume-recommender\evaluation
Fragrantica 콘텐츠: 24,063행, 22열
Atrafshan 평점: 5,565행, 7열
연결된 평점: 3,530행, 8열
향수 ID 매핑: 1,369행, 6열
Atrafshan 향수: 1,369행, 7열

평점 데이터 컬럼:
['user_id', 'perfume_id', 'scent_rating', 'comment_id', 'is_comment_answer', 'source_dataset', 'source_row']

평점 데이터 예시:


,user_id,perfume_id,scent_rating,comment_id,is_comment_answer,source_dataset,source_row
0,atr:user:93040,atr:1030,8.0,128773,NO,atrafshan_data.json,1
1,atr:user:83823,atr:1030,10.0,128366,NO,atrafshan_data.json,2
2,atr:user:70512,atr:1030,1.0,123777,NO,atrafshan_data.json,17
3,atr:user:62518,atr:1030,10.0,122212,NO,atrafshan_data.json,33
4,atr:user:83877,atr:1030,8.0,120794,NO,atrafshan_data.json,38



결측값 확인:
user_id         0
perfume_id      0
scent_rating    0
dtype: int64


In [2]:
# 사용자별 평점 분포 및 평가 가능성 확인
import numpy as np
import pandas as pd

# 사용자 ID를 문자열로 통일
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["perfume_id"] = ratings["perfume_id"].astype(str)
ratings["scent_rating"] = pd.to_numeric(
    ratings["scent_rating"], errors="raise"
)

# 사용자별 평점 개수
user_counts = ratings.groupby("user_id").size()

print("전체 사용자 수:", len(user_counts))
print("전체 평점 수:", len(ratings))
print("사용자당 평점 수 중앙값:", user_counts.median())
print("사용자당 평균 평점 수:", round(user_counts.mean(), 2))

print("\n평점 개수별 사용자 분포")
summary = pd.DataFrame({
    "조건": ["1개", "2개", "3개 이상", "5개 이상", "10개 이상"],
    "사용자 수": [
        (user_counts == 1).sum(),
        (user_counts == 2).sum(),
        (user_counts >= 3).sum(),
        (user_counts >= 5).sum(),
        (user_counts >= 10).sum()
    ]
})
display(summary)

# 공통 카탈로그로 평가할 수 있는 연결 평점 확인
exact = mapping.loc[
    mapping["match_status"].eq("matched_exact")
].copy()

exact_ids = set(exact["atrafshan_id"].astype(str))
linked_eval = linked.loc[
    linked["perfume_id"].astype(str).isin(exact_ids)
    & linked["fragrantica_id"].astype(str).isin(
        perfumes["perfume_id"].astype(str)
    )
].copy()

linked_counts = linked_eval.groupby("user_id").size()

print("\n공통 카탈로그 연결 평점:", len(linked_eval))
print("연결 평점 사용자 수:", len(linked_counts))
print("연결 평점 2개 이상 사용자:", (linked_counts >= 2).sum())
print("연결 평점 3개 이상 사용자:", (linked_counts >= 3).sum())
print("연결 평점 5개 이상 사용자:", (linked_counts >= 5).sum())

print("\n사용자-향수 중복:", ratings.duplicated(
    ["user_id", "perfume_id"]
).sum())
print("평점 범위:", ratings["scent_rating"].min(),
      "~", ratings["scent_rating"].max())

전체 사용자 수: 1747
전체 평점 수: 5565
사용자당 평점 수 중앙값: 1.0
사용자당 평균 평점 수: 3.19

평점 개수별 사용자 분포


,조건,사용자 수
0,1개,947
1,2개,273
2,3개 이상,527
3,5개 이상,290
4,10개 이상,118



공통 카탈로그 연결 평점: 3530
연결 평점 사용자 수: 1355
연결 평점 2개 이상 사용자: 566
연결 평점 3개 이상 사용자: 361
연결 평점 5개 이상 사용자: 180

사용자-향수 중복: 0
평점 범위: 1.0 ~ 10.0


## 2. Train/Test 분리

사용자별 Leave-One-Out 방식으로 테스트 평점을 분리한다.

동일한 사용자–향수 쌍이 학습 데이터와 테스트 데이터에
동시에 포함되지 않도록 중복 여부를 확인한다.

재현성을 위해 RANDOM_STATE = 42를 사용한다.

In [3]:
# 사용자별 Leave-One-Out Train/Test 분리
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 문자열 ID 통일
ratings_split = ratings.copy()
linked_split = linked.copy()

for df in [ratings_split, linked_split]:
    df["user_id"] = df["user_id"].astype(str)
    df["perfume_id"] = df["perfume_id"].astype(str)
    df["scent_rating"] = pd.to_numeric(
        df["scent_rating"], errors="raise"
    )

linked_split["fragrantica_id"] = (
    linked_split["fragrantica_id"].astype(str)
)

# 연결 평점 3개 이상인 사용자
linked_counts = linked_split.groupby("user_id").size()
eligible_users = linked_counts[
    linked_counts >= 3
].index

# 사용자별 테스트 평점 1개 선택
test_indices = []

for user_id in sorted(eligible_users):
    user_rows = linked_split.index[
        linked_split["user_id"] == user_id
    ].to_numpy()

    selected = rng.choice(user_rows)
    test_indices.append(selected)

test_linked = linked_split.loc[test_indices].copy()

# 전체 Atrafshan 평점에서 테스트 쌍 제외
test_pairs = pd.MultiIndex.from_frame(
    test_linked[["user_id", "perfume_id"]]
)

rating_pairs = pd.MultiIndex.from_frame(
    ratings_split[["user_id", "perfume_id"]]
)

test_mask = rating_pairs.isin(test_pairs)

train_ratings = ratings_split.loc[~test_mask].copy()

# 콘텐츠 기반 모델의 학습용 사용자 이력
linked_pairs = pd.MultiIndex.from_frame(
    linked_split[["user_id", "perfume_id"]]
)

train_linked = linked_split.loc[
    ~linked_pairs.isin(test_pairs)
].copy()

# 데이터 누수 확인
train_pairs = pd.MultiIndex.from_frame(
    train_ratings[["user_id", "perfume_id"]]
)

assert not train_pairs.isin(test_pairs).any()
assert len(test_linked) == len(eligible_users)
assert len(train_ratings) + len(test_linked) == len(ratings_split)

print("=== Train/Test 분리 결과 ===")
print("KNN 학습 평점:", len(train_ratings))
print("콘텐츠 사용자 프로필용 평점:", len(train_linked))
print("테스트 평점:", len(test_linked))
print("테스트 대상 사용자:", test_linked.user_id.nunique())

print("\n=== 테스트 평점 예시 ===")
display(test_linked.head())

print("\n=== 데이터 누수 검사 ===")
print("Train/Test 중복 없음")

=== Train/Test 분리 결과 ===
KNN 학습 평점: 5204
콘텐츠 사용자 프로필용 평점: 3169
테스트 평점: 361
테스트 대상 사용자: 361

=== 테스트 평점 예시 ===


,user_id,perfume_id,scent_rating,comment_id,is_comment_answer,source_dataset,source_row,fragrantica_id
1112,atr:user:10391,atr:6825,7.0,77058,NO,atrafshan_data.json,12411,fr:18306
2977,atr:user:10734,atr:111112,8.0,112777,NO,atrafshan_data.json,29962,fr:79632
1995,atr:user:11703,atr:3013,9.0,58489,NO,atrafshan_data.json,20960,fr:31294
1627,atr:user:1195,atr:7366,7.0,86024,NO,atrafshan_data.json,17996,fr:45396
2547,atr:user:13098,atr:4848,9.0,47492,NO,atrafshan_data.json,26521,fr:39314



=== 데이터 누수 검사 ===
Train/Test 중복 없음


In [4]:
# 기존 노트북의 모델 클래스 불러오기
import ast
import json
from pathlib import Path

notebook_path = Path("../perfume_recommendation.ipynb")

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

namespace = {"__name__": "__evaluation__"}

# 기존 노트북의 import와 함수/클래스 정의만 실행
for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    selected = [
        node for node in tree.body
        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        )
    ]

    if not selected:
        continue

    module = ast.Module(body=selected, type_ignores=[])
    ast.fix_missing_locations(module)

    exec(
        compile(module, str(notebook_path), "exec"),
        namespace,
    )

ContentBasedRecommender = namespace[
    "ContentBasedRecommender"
]

ItemKNNRecommender = namespace[
    "ItemKNNRecommender"
]

print("콘텐츠 기반 모델:", ContentBasedRecommender.__name__)
print("협업 필터링 모델:", ItemKNNRecommender.__name__)
print("두 모델 클래스 로딩 완료")

콘텐츠 기반 모델: ContentBasedRecommender
협업 필터링 모델: ItemKNNRecommender
두 모델 클래스 로딩 완료


## 3. 추천 모델 학습

두 가지 추천 모델을 학습한다.

1. Content-based Filtering
   - 향수 콘텐츠 특성 기반 추천

2. Item-based KNN
   - 사용자 평점 이력 기반 협업 필터링

모델별 학습 시간과 사용 데이터를 기록한다.

In [5]:
# 두 추천 모델 학습
import time

# 필요한 변수 확인
required_vars = [
    "perfumes",
    "atrafshan",
    "train_ratings",
    "train_linked",
    "test_linked",
]

missing = [
    name for name in required_vars
    if name not in globals()
]

if missing:
    raise NameError(
        f"이전 셀에서 생성되지 않은 변수: {missing}"
    )

# 1. 콘텐츠 기반 필터링
print("=== 콘텐츠 기반 모델 학습 ===")

start = time.perf_counter()

content_model = ContentBasedRecommender(
    brand_weight=0.2
).fit(perfumes)

content_time = time.perf_counter() - start

print(f"학습 완료: {content_time:.2f}초")


# 2. 메모리 기반 협업 필터링
print("\n=== Item-based KNN 모델 학습 ===")

start = time.perf_counter()

knn_model = ItemKNNRecommender(
    k=20,
    min_common_users=2,
    shrinkage=10.0,
    min_neighbors=1
).fit(
    train_ratings,
    item_metadata=atrafshan
)

knn_time = time.perf_counter() - start

print(f"학습 완료: {knn_time:.2f}초")


# 3. 학습 결과 요약
print("\n=== 학습 결과 ===")
print("콘텐츠 기반 모델: 학습 완료")
print("협업 필터링 모델: 학습 완료")
print(f"콘텐츠 기반 학습 시간: {content_time:.2f}초")
print(f"협업 필터링 학습 시간: {knn_time:.2f}초")
print(f"보류된 테스트 평점: {len(test_linked)}개")

=== 콘텐츠 기반 모델 학습 ===
학습 완료: 0.24초

=== Item-based KNN 모델 학습 ===
학습 완료: 0.02초

=== 학습 결과 ===
콘텐츠 기반 모델: 학습 완료
협업 필터링 모델: 학습 완료
콘텐츠 기반 학습 시간: 0.24초
협업 필터링 학습 시간: 0.02초
보류된 테스트 평점: 361개


In [6]:
# 두 추천 모델의 추천 결과 검증
import pandas as pd

TOP_K = 10
LIKE_THRESHOLD = 7

# 두 모델에서 공통으로 사용할 향수 후보
common = mapping.loc[
    mapping["match_status"].eq("matched_exact")
    & mapping["atrafshan_id"].isin(knn_model.item_ids_)
    & mapping["fragrantica_id"].isin(content_model.item_index_)
].copy()

common = common.drop_duplicates(
    subset=["atrafshan_id", "fragrantica_id"]
)

# 일대일 연결만 사용
common = common.loc[
    ~common["atrafshan_id"].duplicated(keep=False)
    & ~common["fragrantica_id"].duplicated(keep=False)
].copy()

print("공통 후보 향수:", len(common))

# 학습 이력에 존재하고 테스트 평점이 있는 사용자
test_users = set(test_linked["user_id"].astype(str))
train_users = set(train_linked["user_id"].astype(str))
eligible_users = sorted(test_users & train_users)

if not eligible_users:
    raise ValueError("학습·테스트에 공통으로 존재하는 사용자가 없습니다.")

user_id = eligible_users[0]
print("평가 예시 사용자:", user_id)

# 콘텐츠 기반 추천
cb_result = content_model.recommend_for_user(
    user_id=user_id,
    train_linked_ratings=train_linked,
    top_n=TOP_K,
    like_threshold=LIKE_THRESHOLD,
    candidate_ids=common["fragrantica_id"]
)

# 협업 필터링 추천
cf_result = knn_model.recommend(
    user_id=user_id,
    top_n=TOP_K,
    candidate_ids=common["atrafshan_id"],
    include_fallback=False
)

print("\n콘텐츠 기반 추천:", len(cb_result), "개")
display(cb_result.head(10))

print("\n협업 필터링 추천:", len(cf_result), "개")
display(cf_result.head(10))

공통 후보 향수: 666
평가 예시 사용자: atr:user:10391

콘텐츠 기반 추천: 10 개


,perfume_id,score,perfume_name,brand,gender,url
0,fr:15940,0.385918,secret-mission,baldessarini,men,https://www.fragrantica.com/perfume/baldessari...
1,fr:43538,0.348752,momentum-intense,bentley,men,https://www.fragrantica.com/perfume/bentley/mo...
2,fr:258,0.338699,eternity-for-men,calvin-klein,men,https://www.fragrantica.com/perfume/calvin-kle...
3,fr:890,0.327674,polo,ralph-lauren,men,https://www.fragrantica.com/perfume/ralph-laur...
4,fr:598,0.325077,visit-for-men,azzaro,men,https://www.fragrantica.com/perfume/azzaro/vis...
5,fr:614,0.323285,egoiste-platinum,chanel,men,https://www.fragrantica.com/perfume/chanel/ego...
6,fr:53641,0.319706,oud-for-greatness,initio-parfums-prives,unisex,https://www.fragrantica.com/perfume/initio-par...
7,fr:15942,0.317773,vince-camuto-for-men,vince-camuto,men,https://www.fragrantica.com/perfume/vince-camu...
8,fr:430,0.313278,le-male,jean-paul-gaultier,men,https://www.fragrantica.com/perfume/jean-paul-...
9,fr:788,0.312584,chrome,azzaro,men,https://www.fragrantica.com/perfume/azzaro/chr...



협업 필터링 추천: 10 개


,user_id,perfume_id,predicted_rating,method,n_neighbors,perfume_name,brand,url
0,atr:user:10391,atr:10042,10.0,knn,1,Le Male,Jean Paul GAULTIER,https://www.atrafshan.ir//perfume/10042-9543/l...
1,atr:user:10391,atr:11943,10.0,knn,1,Bleu de Chanel Parfum,CHANEL,https://www.atrafshan.ir//perfume/11943-11787/...
2,atr:user:10391,atr:13762,10.0,knn,1,Angels' Share,by Kilian,https://www.atrafshan.ir//perfume/13762-64419/...
3,atr:user:10391,atr:1382,10.0,knn,1,Tobacco Vanille,TOM FORD,https://www.atrafshan.ir//perfume/1382-1274/to...
4,atr:user:10391,atr:14993,10.0,knn,1,Sauvage Elixir,Dior,https://www.atrafshan.ir//perfume/14993-16300/...
5,atr:user:10391,atr:1504,10.0,knn,1,Herod,PARFUMS de MARLY,https://www.atrafshan.ir//perfume/1504-792/herod
6,atr:user:10391,atr:15700,10.0,knn,1,Cuoium,ORTO PARISI,https://www.atrafshan.ir//perfume/15700-17176/...
7,atr:user:10391,atr:1933,10.0,knn,1,Dior Sauvage,Dior,https://www.atrafshan.ir//perfume/1933-1267/di...
8,atr:user:10391,atr:2032,10.0,knn,1,Miss Dior Blooming Bouquet,Dior,https://www.atrafshan.ir//perfume/2032-1371/mi...
9,atr:user:10391,atr:2431,10.0,knn,1,Cedrat Boise,MANCERA,https://www.atrafshan.ir//perfume/2431-1838/ce...


In [7]:
# 전체 테스트 사용자 추천 가능 여부 및 KNN 신뢰도 점검
import pandas as pd

TOP_K = 10

test_users = sorted(test_linked["user_id"].astype(str).unique())

records = []
neighbor_counts = []

for user_id in test_users:
    cb = content_model.recommend_for_user(
        user_id=user_id,
        train_linked_ratings=train_linked,
        top_n=TOP_K,
        like_threshold=7,
        candidate_ids=common["fragrantica_id"]
    )

    cf = knn_model.recommend(
        user_id=user_id,
        top_n=TOP_K,
        candidate_ids=common["atrafshan_id"],
        include_fallback=False
    )

    records.append({
        "user_id": user_id,
        "content_count": len(cb),
        "knn_count": len(cf),
        "content_available": len(cb) > 0,
        "knn_available": len(cf) > 0
    })

    if not cf.empty:
        neighbor_counts.extend(
            cf["n_neighbors"].tolist()
        )

availability = pd.DataFrame(records)

print("전체 테스트 사용자:", len(availability))
print(
    "콘텐츠 기반 추천 가능:",
    availability["content_available"].sum()
)
print(
    "KNN 추천 가능:",
    availability["knn_available"].sum()
)
print(
    "두 모델 모두 추천 가능:",
    (
        availability["content_available"]
        & availability["knn_available"]
    ).sum()
)

print("\n모델별 추천 개수 통계")
display(
    availability[
        ["content_count", "knn_count"]
    ].describe()
)

print("\nKNN 이웃 수 분포")
if neighbor_counts:
    display(
        pd.Series(neighbor_counts, name="n_neighbors")
        .value_counts()
        .sort_index()
        .rename_axis("이웃 수")
        .reset_index(name="추천 건수")
    )
else:
    print("KNN 추천 결과 없음")

전체 테스트 사용자: 361
콘텐츠 기반 추천 가능: 352
KNN 추천 가능: 352
두 모델 모두 추천 가능: 343

모델별 추천 개수 통계


,content_count,knn_count
count,361.000000,361.000000
mean,9.750693,9.130194
std,1.561305,2.359897
min,0.000000,0.000000
25%,10.000000,10.000000
50%,10.000000,10.000000
75%,10.000000,10.000000
max,10.000000,10.000000



KNN 이웃 수 분포


,이웃 수,추천 건수
0,1,2584
1,2,588
2,3,96
3,4,19
4,5,5
5,6,3
6,7,1


In [8]:
# 테스트 정답 데이터 및 평가 조건 점검
import pandas as pd

RELEVANCE_THRESHOLD = 7.0

test = test_linked.copy()
train = train_linked.copy()

for df in [train, test]:
    df["user_id"] = df["user_id"].astype(str)
    df["perfume_id"] = df["perfume_id"].astype(str)
    df["scent_rating"] = pd.to_numeric(
        df["scent_rating"], errors="coerce"
    )

# 테스트 평점 통계
print("=== 테스트 평점 통계 ===")
print("전체 테스트 평점:", len(test))
print("테스트 사용자:", test["user_id"].nunique())
print("테스트 향수:", test["perfume_id"].nunique())

print("\n평점 분포")
display(test["scent_rating"].describe())

# 관련 아이템(좋아한 향수) 정의
relevant = test[
    test["scent_rating"] >= RELEVANCE_THRESHOLD
].copy()

print("\n=== 관련 아이템 기준 ===")
print("기준 평점:", RELEVANCE_THRESHOLD)
print("관련 테스트 평점:", len(relevant))
print("관련 아이템이 있는 사용자:",
      relevant["user_id"].nunique())

# Train/Test 중복 확인
train_pairs = set(zip(
    train["user_id"], train["perfume_id"]
))
test_pairs = set(zip(
    test["user_id"], test["perfume_id"]
))

overlap = train_pairs & test_pairs

print("\n=== 데이터 누수 점검 ===")
print("학습/테스트 중복 사용자-향수 쌍:", len(overlap))

# 테스트 사용자 학습 이력 확인
train_users = set(train["user_id"])
test_users = set(test["user_id"])

print("\n=== 평가 사용자 확인 ===")
print("학습 이력이 있는 테스트 사용자:",
      len(train_users & test_users))
print("학습 이력이 없는 테스트 사용자:",
      len(test_users - train_users))

=== 테스트 평점 통계 ===
전체 테스트 평점: 361
테스트 사용자: 361
테스트 향수: 231

평점 분포


count    361.000000
mean       7.637119
std        1.920259
min        1.000000
25%        7.000000
50%        8.000000
75%        9.000000
max       10.000000
Name: scent_rating, dtype: float64


=== 관련 아이템 기준 ===
기준 평점: 7.0
관련 테스트 평점: 286
관련 아이템이 있는 사용자: 286

=== 데이터 누수 점검 ===
학습/테스트 중복 사용자-향수 쌍: 0

=== 평가 사용자 확인 ===
학습 이력이 있는 테스트 사용자: 361
학습 이력이 없는 테스트 사용자: 0


In [9]:
# 공통 평가 가능 사용자 중 관련 테스트 항목이 있는 사용자 확인
RELEVANCE_THRESHOLD = 7.0

common_users = set(
    availability.loc[
        availability["content_available"]
        & availability["knn_available"],
        "user_id"
    ].astype(str)
)

relevant_users = set(
    test_linked.loc[
        pd.to_numeric(
            test_linked["scent_rating"],
            errors="coerce"
        ) >= RELEVANCE_THRESHOLD,
        "user_id"
    ].astype(str)
)

evaluation_users = common_users & relevant_users

print("=== 평가 대상 사용자 ===")
print("공통 추천 가능 사용자:", len(common_users))
print("관련 테스트 항목 보유 사용자:", len(relevant_users))
print("최종 평가 후보 사용자:", len(evaluation_users))

=== 평가 대상 사용자 ===
공통 추천 가능 사용자: 343
관련 테스트 항목 보유 사용자: 286
최종 평가 후보 사용자: 271


## 4. Top-10 추천 성능 평가

두 모델에 동일한 평가 조건을 적용한다.

- 추천 개수: Top-10
- 관련 아이템 기준: 평점 7점 이상
- 평가 지표: Hit Rate@10, Precision@10, Recall@10

추천 결과와 테스트 정답을 비교하여
사용자별 지표를 계산하고 평균을 산출한다.

In [10]:
# 두 추천 모델의 공통 후보 기준 Top-10 성능 평가
import pandas as pd

TOP_K = 10
RELEVANCE_THRESHOLD = 7.0

# Atrafshan ID와 Fragrantica ID의 일대일 매핑
atr_to_fr = dict(
    zip(common["atrafshan_id"].astype(str),
        common["fragrantica_id"].astype(str))
)

candidate_atr = set(atr_to_fr.keys())

# 테스트 정답 중 공통 후보 카탈로그에 속한 항목
test_eval = test_linked.copy()
test_eval["user_id"] = test_eval["user_id"].astype(str)
test_eval["perfume_id"] = test_eval["perfume_id"].astype(str)
test_eval["scent_rating"] = pd.to_numeric(
    test_eval["scent_rating"], errors="coerce"
)

test_eval = test_eval[
    test_eval["perfume_id"].isin(candidate_atr)
].copy()

# 두 모델 모두 추천 가능한 사용자 중 테스트 정답이 있는 사용자
eval_users = sorted(
    set(test_eval["user_id"]) & common_users
)

records = []

for user_id in eval_users:
    truth = set(
        test_eval.loc[
            (test_eval["user_id"] == user_id)
            & (test_eval["scent_rating"] >= RELEVANCE_THRESHOLD),
            "perfume_id"
        ]
    )

    # 관련 정답이 없는 사용자는 별도 집계
    if not truth:
        continue

    # 콘텐츠 기반 추천
    cb = content_model.recommend_for_user(
        user_id=user_id,
        train_linked_ratings=train_linked,
        top_n=TOP_K,
        like_threshold=7,
        candidate_ids=common["fragrantica_id"]
    )

    # 협업 필터링 추천
    cf = knn_model.recommend(
        user_id=user_id,
        top_n=TOP_K,
        candidate_ids=common["atrafshan_id"],
        include_fallback=False
    )

    # 콘텐츠 기반 결과를 Atrafshan ID로 변환
    fr_to_atr = {v: k for k, v in atr_to_fr.items()}

    cb_ids = [
        fr_to_atr[str(pid)]
        for pid in cb["perfume_id"].astype(str).head(TOP_K)
        if str(pid) in fr_to_atr
    ]

    cf_ids = (
        cf["perfume_id"].astype(str).head(TOP_K).tolist()
    )

    for model_name, recommendations in [
        ("Content-based", cb_ids),
        ("Item-based KNN", cf_ids)
    ]:
        hits = len(set(recommendations) & truth)

        records.append({
            "user_id": user_id,
            "model": model_name,
            "hits": hits,
            "hit_rate@10": float(hits > 0),
            "precision@10": hits / TOP_K,
            "recall@10": hits / len(truth)
        })

results = pd.DataFrame(records)

if results.empty:
    raise ValueError("평가 가능한 사용자가 없습니다.")

summary = (
    results.groupby("model")
    .agg(
        users=("user_id", "nunique"),
        hit_rate_at_10=("hit_rate@10", "mean"),
        precision_at_10=("precision@10", "mean"),
        recall_at_10=("recall@10", "mean")
    )
    .round(4)
)

print("공통 후보 향수:", len(common))
print("최종 평가 사용자:", len(eval_users))
print("실제 지표 계산 사용자:", results["user_id"].nunique())

display(summary)

# 평가 결과 저장
from pathlib import Path

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

summary.to_csv(
    output_dir / "model_comparison.csv",
    encoding="utf-8-sig"
)

print("모델 성능 비교 결과 저장 완료")

공통 후보 향수: 666
최종 평가 사용자: 326
실제 지표 계산 사용자: 258


,users,hit_rate_at_10,precision_at_10,recall_at_10
model,,,,
Content-based,258,0.0620,0.0062,0.0620
Item-based KNN,258,0.0581,0.0058,0.0581


모델 성능 비교 결과 저장 완료


## 5. 결과 분석

현재 평가 결과에서는 Content-based 모델이
Item-based KNN보다 소폭 높은 성능을 기록하였다.

단, 데이터 희소성, 향수 ID 매핑 범위,
추천 후보 제한 및 평가 사용자 선정 조건이
성능에 영향을 줄 수 있다.

상세한 평가 방법과 한계는 EVALUATION.md를 참고한다.